# Paper §6.2 — Cross-dataset anchoring reduction (chosen cell paired-bootstrap CI)

Reproduces the §6.2 headline: at the chosen cell `L=26 K=8 α=1.0` on OneVision Main, the calibrated subspace projection reduces anchoring across 5 datasets — **5/5 Δem(b) sign-clean under both 95 % and Bonferroni-20** corrected paired-bootstrap CIs. Inference is one cell × 5 datasets (PlotQA / InfoVQA / TallyQA / ChartQA / MathVista). Aggregator = `build_e6_stage4_bootstrap_ci.py`; figure = `build_paper_stage4_paired_ci_figure.py`.

**Spec source-of-truth.** `docs/paper/emnlp_outline_ko.md` — §6.2 (Cross-dataset anchoring reduction).

This notebook drives heavy inference stages by `subprocess`-invoking the
existing drivers in `scripts/`. The `RUN_INFERENCE = False` toggle below
lets a reviewer read the full pipeline without GPU access — canonical
CSVs are read straight from disk and figures get re-rendered from them.


## 1 · Setup — paths + subprocess helper

In [ ]:
from __future__ import annotations
import json
import os
import shlex
import subprocess
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def find_main_worktree() -> Path:
    common = subprocess.check_output(
        ["git", "rev-parse", "--git-common-dir"], cwd=Path.cwd(), text=True
    ).strip()
    return Path(common).resolve().parent


def find_worktree_root() -> Path:
    return Path(subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"], cwd=Path.cwd(), text=True
    ).strip()).resolve()


MAIN     = find_main_worktree()
WORKTREE = find_worktree_root()

# Scripts + configs from the active branch (worktree); gitignored artifacts
# (inputs/, outputs/, docs/insights/_data/) from MAIN.
SCRIPTS    = WORKTREE / "scripts"
CONFIGS    = WORKTREE / "configs"
DATA_DIR   = MAIN / "docs" / "insights" / "_data"
FIGURES    = WORKTREE / "docs" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

GPUS = os.environ.get("VLM_ANCHOR_GPUS", "0,1,2,3,4")
RUN_INFERENCE = False  # set True to invoke the heavy driver(s)

print(f"MAIN     = {MAIN}")
print(f"WORKTREE = {WORKTREE}")
print(f"GPUS     = {GPUS}")
print(f"RUN_INFERENCE = {RUN_INFERENCE}")


In [ ]:
def run_cmd(cmd: list[str] | str, *, dry: bool = False, env: dict | None = None) -> int:
    printable = " ".join(shlex.quote(c) for c in cmd) if isinstance(cmd, list) else cmd
    print(f"$ {printable}")
    if dry:
        print("  (dry — RUN_INFERENCE=False)")
        return 0
    full_env = os.environ.copy()
    if env:
        full_env.update(env)
    return subprocess.run(cmd, cwd=MAIN, env=full_env,
                          shell=isinstance(cmd, str)).returncode


def save_figure(fig, stem: str, png_dir: Path = FIGURES,
                pdf_dir: Path | None = None):
    png = png_dir / f"{stem}.png"
    fig.savefig(png, bbox_inches="tight", dpi=160)
    print(f"wrote {png}")
    if pdf_dir is not None:
        pdf_dir.mkdir(parents=True, exist_ok=True)
        pdf = pdf_dir / f"{stem}.pdf"
        fig.savefig(pdf, bbox_inches="tight")
        print(f"wrote {pdf}")


## 2 · Chosen cell — single-cell sweep across 5 datasets

For each dataset we run a one-cell sweep with `L=26, K=8, α=1.0` on top
of the canonical pooled subspace. Output dir is
`sweep_subspace_<ds_tag>_<scope>_chosen/`, which is what
`build_e6_stage4_bootstrap_ci.py` reads.

Note that the §5.2 reproduction already swept the same chosen cell as
part of its `..._p4_layer_sweep_K1_layers_K8` dir (one cell among many).
If you already ran §5.2, the same predictions can be reused — see the
`reuse_section_5_2` switch below.


In [ ]:
ONEVISION    = "llava-onevision-qwen2-7b-ov"
ONEVISION_HF = "llava-hf/llava-onevision-qwen2-7b-ov-hf"

SUBSPACE_PATH = MAIN / "outputs" / "e6_steering" / ONEVISION / "_subspace" / "subspace_plotqa_infovqa_pooled_n5k_K16.pt"
SCOPE         = "plotqa_infovqa_pooled_n5k"

CHOSEN_DATASETS = [
    ("plotqa",         "experiment_e7_plotqa_full",
     "outputs/experiment_e7_plotqa_full/llava-onevision-qwen2-7b-ov/20260502-132624/predictions.jsonl"),
    ("infographicvqa", "experiment_e7_infographicvqa_full",
     "outputs/experiment_e7_infographicvqa_full/llava-onevision-qwen2-7b-ov/20260502-152105/predictions.jsonl"),
    ("tallyqa",        "experiment_e5e_tallyqa_full",
     "outputs/experiment_e5e_tallyqa_full/llava-onevision-qwen2-7b-ov/20260502-083926/predictions.jsonl"),
    ("chartqa",        "experiment_e5e_chartqa_full",
     "outputs/experiment_e5e_chartqa_full/llava-onevision-qwen2-7b-ov/20260502-211028/predictions.jsonl"),
    ("mathvista",      "experiment_e5e_mathvista_full",
     "outputs/experiment_e5e_mathvista_full/llava-onevision-qwen2-7b-ov/20260502-212440/predictions.jsonl"),
]


def sweep_chosen(ds_tag: str, cfg: str, pred: str):
    out_dir = MAIN / "outputs" / "e6_steering" / ONEVISION / f"sweep_subspace_{ds_tag}_{SCOPE}_chosen"
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "run_sweep_subspace_sharded.py"),
        "--config", str(CONFIGS / f"{cfg}.yaml"),
        "--model", ONEVISION, "--hf-model", ONEVISION_HF,
        "--predictions-path", str(MAIN / pred),
        "--dataset-tag", ds_tag,
        "--subspace-path", str(SUBSPACE_PATH),
        "--subspace-scope", SCOPE,
        "--sweep-layers", "26",
        "--sweep-alphas", "1.0",
        "--sweep-ks", "8",
        "--batch-size", "1",
        "--prefetch-workers", "16",
        "--gpus", GPUS,
        "--out-dir", str(out_dir),
    ]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


for ds_tag, cfg, pred in CHOSEN_DATASETS:
    sweep_chosen(ds_tag, cfg, pred)


## 3 · Stage-4 paired-bootstrap CI aggregation

`build_e6_stage4_bootstrap_ci.py` reads each
`sweep_subspace_<ds>_<scope>_chosen/predictions.jsonl`, paired-resamples
sids B=10,000 times, and writes both 95 % and Bonferroni-20 corrected CIs
on Δadopt, Δdf, Δem(a), Δem(b) per dataset.


In [ ]:
def aggregate_stage4(bootstrap: int = 10_000, seed: int = 20260510):
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "build_e6_stage4_bootstrap_ci.py"),
        "--bootstrap", str(bootstrap),
        "--seed", str(seed),
        "--out-dir", str(DATA_DIR),
    ]
    return run_cmd(cmd, dry=not RUN_INFERENCE)


aggregate_stage4()


## 4 · Table 6.2 — Per-dataset Δ at chosen cell

Reads the canonical stage4 CSV and emits Table 6.2 in markdown form.


In [ ]:
stage4_csv = DATA_DIR / "stage4_final_per_dataset_ci.csv"
if stage4_csv.exists():
    df = pd.read_csv(stage4_csv)
    rows = []
    for ds, label, scope in [
        ("TallyQA", "TallyQA", "held-out"),
        ("PlotQA", "PlotQA", "within"),
        ("InfoVQA", "InfoVQA", "within"),
        ("ChartQA", "ChartQA", "held-out"),
        ("MathVista", "MathVista", "held-out"),
    ]:
        sub = df[df["dataset"] == ds]
        if not len(sub):
            continue
        r = sub.iloc[0]
        rows.append({
            "Dataset": label,
            "n": int(r["n_paired"]),
            "Δ adopt(a)": f"{r['delta_adopt']*100:+.1f} [{r['delta_adopt_ci95_lo']*100:+.1f}, {r['delta_adopt_ci95_hi']*100:+.1f}]",
            "Δ df(a)":    f"{r['delta_df']*100:+.1f} [{r['delta_df_ci95_lo']*100:+.1f}, {r['delta_df_ci95_hi']*100:+.1f}]",
            "Δ em(a)":    f"{r['delta_em_a']*100:+.1f} [{r['delta_em_a_ci95_lo']*100:+.1f}, {r['delta_em_a_ci95_hi']*100:+.1f}]",
            "Δ em(b)":    f"{r['delta_em_b']*100:+.1f} [{r['delta_em_b_ci95_lo']*100:+.1f}, {r['delta_em_b_ci95_hi']*100:+.1f}]",
            "scope":      scope,
        })
    out = pd.DataFrame(rows)
    print(out.to_string(index=False))
else:
    print(f"missing: {stage4_csv}")


## 5 · Figure §6.2.3 — paired-bootstrap forest panels


In [ ]:
def render_stage4_figure():
    cmd = [
        "uv", "run", "python", str(SCRIPTS / "build_paper_stage4_paired_ci_figure.py"),
    ]
    return run_cmd(cmd, dry=False)


render_stage4_figure()

# Display the figure
fig_path = WORKTREE / "docs" / "figures" / "paper_6_2_3_stage4_5dataset_paired_ci.png"
if fig_path.exists():
    from IPython.display import Image, display
    display(Image(str(fig_path)))
else:
    print(f"missing: {fig_path}")


## Summary

Pipeline:
1. 5-dataset chosen-cell sweep (`L=26 K=8 α=1.0`, OneVision Main) →
   `sweep_subspace_<ds>_<scope>_chosen/predictions.jsonl`
2. `build_e6_stage4_bootstrap_ci.py` paired-bootstrap (B=10,000) →
   `docs/insights/_data/stage4_final_per_dataset_ci.{csv,md}`
3. `build_paper_stage4_paired_ci_figure.py` →
   `docs/figures/paper_6_2_3_stage4_5dataset_paired_ci.png`

Headline numbers (paper §6.2.2, Δem(b) sign-clean column):
- TallyQA  +13.8 [+12.9, +14.8]  (held-out)
- PlotQA    +4.7 [+3.8, +5.7]    (within calibration distribution)
- InfoVQA   +9.0 [+6.3, +11.7]   (within)
- ChartQA   +7.1 [+3.6, +10.7]   (held-out)
- MathVista +9.4 [+4.7, +14.7]   (held-out)

5/5 sign-clean under both 95 % and Bonferroni-20 corrected CIs.

§6.2 numbers are also reproducible from the §5.2 reproduction's
`outputs/paper/section_5_e6_steering/_data/p4_layer_sweep_per_cell_ci.csv`
at the chosen-cell row (L=26 K=8 α=1.0) for each dataset — match within
bf16 precision.
